# SEC-C Mini-GAT V2 GNN Training Notebook

**Model:** MiniGATv2 (2-layer GAT, 4 heads each, 774-dim input)

**Improvements over V1:**
- Multi-language: Python, JavaScript, Java, C/C++, Go (was Java+C only)
- Real-world datasets: CVEfixes, DiverseVul, Devign (was Juliet-only)
- Balanced classes: max 2:1 vuln:safe ratio (was 13:1)
- Improved graph construction: AST+CFG+DDG edges (was sequential-only)
- Focal Loss (was weighted CE)
- 774-dim features: 768 GraphCodeBERT + 6 structural (was 773 = 768+5)
- Conformal prediction: alpha=0.3 (was 0.1 -> threshold=1.0 problem)
- 4-way split: train/val/cal/test (separate calibration set)

## Cell 1: Setup & Install

In [1]:
# ============================================================================
# Cell 1: Setup & Install Dependencies
# ============================================================================
import subprocess, sys, os

def run_cmd(cmd):
    """Run a shell command, print output."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip()[-500:])
    if result.returncode != 0 and result.stderr.strip():
        print(f"  WARN: {result.stderr.strip()[-300:]}")
    return result.returncode

# Install PyTorch Geometric and deps
import torch
torch_version = torch.__version__.split('+')[0]
cuda_tag = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

print(f"PyTorch: {torch_version}, CUDA tag: {cuda_tag}")

# Install PyG from wheels
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_tag}.html"
run_cmd(f"{sys.executable} -m pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f {pyg_url}")
run_cmd(f"{sys.executable} -m pip install -q torch-geometric")
run_cmd(f"{sys.executable} -m pip install -q transformers datasets scikit-learn matplotlib networkx tqdm")

# GPU check
print("\n" + "="*60)
try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        gpu_mem = props.total_memory / 1e9  # FIX: total_memory not total_mem
        print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    else:
        print("  WARNING: No GPU detected! Training will be very slow.")
except Exception as _gpu_err:
    print(f"  GPU info unavailable: {_gpu_err}")
print("="*60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

PyTorch: 2.10.0, CUDA tag: 128
$ /usr/bin/python3 -m pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 50.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.5 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q torch-geometric
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q transformers datasets scikit-learn matplotlib networkx tqdm

  GPU: Tesla T4 (15.6 GB)
  Device: cuda


## Cell 2: Configuration

In [2]:
# ============================================================================
# Cell 2: Configuration -- all hyperparameters in one place
# ============================================================================
from pathlib import Path
import json

CONFIG = {
    # Model architecture
    "embedding_model": "microsoft/graphcodebert-base",
    "embedding_dim": 768,
    "node_feature_dim": 6,       # was 5 -- added language_id
    "input_dim": 774,            # was 773 -- 768 + 6
    "hidden_dim": 256,
    "output_dim": 128,
    "num_heads_l1": 4,
    "num_heads_l2": 4,
    "dropout": 0.3,
    "num_classes": 2,

    # Data processing
    "max_nodes": 300,            # was 200
    "max_tokens_per_node": 128,
    "embedding_batch_size": 64,

    # Training
    "batch_size": 32,
    "epochs": 60,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "patience": 15,              # was 10
    "confidence_loss_weight": 0.2,
    "grad_clip": 1.0,
    "focal_gamma": 2.0,         # NEW: focal loss gamma

    # Conformal prediction
    "alpha": 0.3,               # was 0.1

    # Data split (4-way)
    "train_ratio": 0.60,
    "val_ratio": 0.15,
    "cal_ratio": 0.15,
    "test_ratio": 0.10,

    # Dataset balancing
    "target_balance_ratio": 2.0,  # max 2:1 vuln:safe

    # Language map
    "language_ids": {
        "python": 0.0,
        "javascript": 0.2,
        "java": 0.4,
        "c_cpp": 0.6,
        "go": 0.8,
    },
}

# Sink patterns for is_sink feature
SINK_PATTERNS = [
    "execute", "exec", "system", "popen", "runtime", "processbuilder",
    "sendredirect", "forward", "include", "write", "print", "println",
    "printf", "sprintf", "fprintf", "strcpy", "strcat", "memcpy",
    "gets", "scanf", "fscanf", "sscanf", "fread", "recv",
    "malloc", "calloc", "realloc", "free",
    "fopen", "open", "connect", "bind", "listen", "accept",
    "eval", "innerhtml", "document.write", "setattribute",
    "preparestatement", "createquery", "executequery", "executeupdate",
    # Python sinks
    "subprocess", "os.system", "os.popen", "pickle.loads", "yaml.load",
    "cursor.execute", "render_template_string",
    # JS sinks
    "child_process", "vm.runinnewcontext", "res.send", "res.write",
    # Go sinks
    "exec.command", "db.query", "db.exec", "fmt.fprintf",
    "http.get", "ioutil.readall",
]

# Source patterns for is_source feature
SOURCE_PATTERNS = [
    "request", "getparameter", "getquerystring", "getheader",
    "getinputstream", "getreader", "getcookies", "getpathinfo",
    "input", "argv", "stdin", "environ", "getenv",
    "args", "fgets", "fread", "recv", "recvfrom",
    "read", "readline", "readlines", "scanner",
    "bufferedreader", "fileinputstream",
    "socket", "urlconnection", "httpservletrequest",
    # Python sources
    "flask.request", "django.request", "sys.argv", "os.environ",
    # JS sources
    "req.body", "req.params", "req.query", "process.env", "process.argv",
    # Go sources
    "r.formvalue", "r.url.query", "os.args", "r.body",
]

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Model: MiniGATv2 ({CONFIG['input_dim']} -> {CONFIG['hidden_dim']} -> {CONFIG['output_dim']})")
print(f"  Features: {CONFIG['embedding_dim']} GCB + {CONFIG['node_feature_dim']} structural = {CONFIG['input_dim']}")
print(f"  Max nodes: {CONFIG['max_nodes']}")
print(f"  Focal gamma: {CONFIG['focal_gamma']}")
print(f"  Conformal alpha: {CONFIG['alpha']}")
print(f"  Languages: {list(CONFIG['language_ids'].keys())}")

Configuration loaded.
  Model: MiniGATv2 (774 -> 256 -> 128)
  Features: 768 GCB + 6 structural = 774
  Max nodes: 300
  Focal gamma: 2.0
  Conformal alpha: 0.3
  Languages: ['python', 'javascript', 'java', 'c_cpp', 'go']


## Cell 3: Download Multi-Language Datasets

In [3]:
# ============================================================================
# Cell 3: Download Multi-Language Datasets
# ============================================================================
import hashlib
import random
import re
import zipfile
import io
import time
from collections import Counter

random.seed(42)

all_samples = []  # List of {code, label, language, cwe, source}

def add_samples(samples, source_name):
    """Add samples with source tag and print count."""
    for s in samples:
        s["source"] = source_name
    all_samples.extend(samples)
    langs = Counter(s["language"] for s in samples)
    labels = Counter(s["label"] for s in samples)
    print(f"  [{source_name}] Added {len(samples)} samples: {dict(labels)}, langs={dict(langs)}")

# -------------------------------------------------------------------------
# 1. CVEfixes (multi-language CSV: code, language, safety)
#    Kaggle: girish17019/cvefixes-vulnerable-and-fixed-code
#    HuggingFace: DetectVul/CVEFixes (Python statement-level)
# -------------------------------------------------------------------------
print("="*60)
print("  1. CVEfixes Dataset")
print("="*60)

cvefixes_loaded = False
try:
    # Try Kaggle pre-attached path
    kaggle_paths = [
        Path("/kaggle/input/cvefixes-vulnerable-and-fixed-code/CVEFixes.csv"),
        Path("/kaggle/input/cvefixes/CVEFixes.csv"),
    ]
    csv_path = None
    for p in kaggle_paths:
        if p.exists():
            csv_path = p
            break

    if csv_path is None:
        # Try Kaggle API
        print("  Trying Kaggle API download...")
        import subprocess
        r = subprocess.run(
            "kaggle datasets download -d girish17019/cvefixes-vulnerable-and-fixed-code -p /tmp/cvefixes --unzip",
            shell=True, capture_output=True, timeout=300
        )
        candidate = Path("/tmp/cvefixes/CVEFixes.csv")
        if candidate.exists():
            csv_path = candidate

    if csv_path is not None:
        print(f"  Loading CSV from {csv_path}...")
        import pandas as pd
        df = pd.read_csv(csv_path, on_bad_lines='skip', engine='python')
        print(f"  Columns: {list(df.columns)}, Shape: {df.shape}")

        lang_map = {
            'python': 'python', 'py': 'python',
            'javascript': 'javascript', 'js': 'javascript',
            'java': 'java',
            'c': 'c_cpp', 'c++': 'c_cpp', 'cpp': 'c_cpp', 'c/c++': 'c_cpp',
            'go': 'go', 'golang': 'go',
        }
        cvefixes_samples = []
        for _, row in df.iterrows():
            code_text = str(row.get('code', ''))
            lang_raw = str(row.get('language', '')).strip().lower()
            safety = str(row.get('safety', '')).strip().lower()
            lang = lang_map.get(lang_raw, None)
            if lang is None or len(code_text.strip().split('\n')) < 3:
                continue
            label = 1 if safety == 'vulnerable' else 0
            cvefixes_samples.append({
                "code": code_text, "label": label,
                "language": lang, "cwe": "CWE-unknown",
            })
        if cvefixes_samples:
            if len(cvefixes_samples) > 5000:
                random.shuffle(cvefixes_samples)
                cvefixes_samples = cvefixes_samples[:5000]
            add_samples(cvefixes_samples, "CVEfixes")
            cvefixes_loaded = True

    if not cvefixes_loaded:
        # Fallback: HuggingFace (Python-only, statement-level)
        print("  Trying HuggingFace DetectVul/CVEFixes...")
        from datasets import load_dataset
        ds = load_dataset("DetectVul/CVEFixes", split="train", streaming=True)
        hf_samples = []
        for i, row in enumerate(ds):
            if i >= 2000:
                break
            raw_lines = row.get("raw_lines", [])
            labels = row.get("label", [])
            if raw_lines and len(raw_lines) >= 3:
                code_text = "\n".join(raw_lines)
                is_vuln = 1 if any(l == 1 for l in labels) else 0
                hf_samples.append({
                    "code": code_text, "label": is_vuln,
                    "language": "python", "cwe": "CWE-unknown",
                })
        if hf_samples:
            add_samples(hf_samples, "CVEfixes-HF")
            cvefixes_loaded = True
except Exception as e:
    print(f"  CVEfixes failed: {e}")

if not cvefixes_loaded:
    print("  CVEfixes not available; will use synthetic fallback")

# -------------------------------------------------------------------------
# 2. DiverseVul (C/C++ functions, HuggingFace: bstee615/diversevul)
# -------------------------------------------------------------------------
print("\n" + "="*60)
print("  2. DiverseVul Dataset (C/C++)")
print("="*60)

diversevul_loaded = False
try:
    from datasets import load_dataset
    print("  Trying HuggingFace bstee615/diversevul (streaming)...")
    ds = load_dataset("bstee615/diversevul", split="train", streaming=True)
    dv_samples = []
    for i, row in enumerate(ds):
        if i >= 3000:
            break
        func_code = row.get("func", "")
        target = row.get("target", 0)
        cwe_list = row.get("cwe", [])
        cwe_str = cwe_list[0] if cwe_list and len(cwe_list) > 0 else "CWE-unknown"
        if len(func_code.strip().split('\n')) < 3:
            continue
        dv_samples.append({
            "code": func_code, "label": int(target),
            "language": "c_cpp", "cwe": str(cwe_str),
        })
    if dv_samples:
        add_samples(dv_samples, "DiverseVul")
        diversevul_loaded = True
except Exception as e:
    print(f"  DiverseVul HF failed: {e}")

if not diversevul_loaded:
    try:
        # Kaggle fallback
        import glob as glob_mod
        for kp in [Path("/kaggle/input/diversevul-json"), Path("/kaggle/input/diversevul")]:
            if kp.exists():
                jf = glob_mod.glob(str(kp / "**/*.json"), recursive=True)
                if jf:
                    with open(jf[0], 'r') as f:
                        dv_data = json.load(f)
                    if isinstance(dv_data, list):
                        samples = []
                        for item in dv_data[:3000]:
                            func = item.get("func", item.get("code", ""))
                            tgt = item.get("target", item.get("label", 0))
                            if len(func.strip().split('\n')) >= 3:
                                samples.append({
                                    "code": func, "label": int(tgt),
                                    "language": "c_cpp",
                                    "cwe": str(item.get("cwe", "CWE-unknown")),
                                })
                        if samples:
                            add_samples(samples, "DiverseVul-Kaggle")
                            diversevul_loaded = True
                break
    except Exception as e2:
        print(f"  DiverseVul Kaggle failed: {e2}")

if not diversevul_loaded:
    print("  DiverseVul not available")

# -------------------------------------------------------------------------
# 3. Devign (C/C++ functions, HuggingFace: DetectVul/devign)
# -------------------------------------------------------------------------
print("\n" + "="*60)
print("  3. Devign Dataset (C/C++)")
print("="*60)

devign_loaded = False
try:
    from datasets import load_dataset
    print("  Trying HuggingFace DetectVul/devign (streaming)...")
    ds = load_dataset("DetectVul/devign", split="train", streaming=True)
    dev_samples = []
    for i, row in enumerate(ds):
        if i >= 2000:
            break
        func_code = row.get("func", "")
        target = row.get("target", False)
        if len(func_code.strip().split('\n')) < 3:
            continue
        dev_samples.append({
            "code": func_code, "label": 1 if target else 0,
            "language": "c_cpp", "cwe": "CWE-unknown",
        })
    if dev_samples:
        add_samples(dev_samples, "Devign")
        devign_loaded = True
except Exception as e:
    print(f"  Devign failed: {e}")

if not devign_loaded:
    print("  Devign not available")

# -------------------------------------------------------------------------
# 4. Juliet Test Suite (Java + C/C++)
# -------------------------------------------------------------------------
print("\n" + "="*60)
print("  4. Juliet Test Suite (Java + C/C++)")
print("="*60)

juliet_loaded = False
try:
    import urllib.request
    juliet_java_url = "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-java-v1-3.zip"
    juliet_c_url = "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3.zip"
    CWE_PATTERN = re.compile(r'CWE(\d+)')

    def extract_juliet(zip_path, language, max_samples=1000):
        samples = []
        ext = '.java' if language == 'java' else '.c'
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                names = [n for n in zf.namelist()
                         if n.endswith(ext) and '/testcases/' in n]
                random.shuffle(names)
                for name in names[:max_samples * 3]:
                    try:
                        content = zf.read(name).decode('utf-8', errors='ignore')
                    except Exception:
                        continue
                    if len(content.strip().split('\n')) < 5:
                        continue
                    cwe_m = CWE_PATTERN.search(name)
                    cwe = f"CWE-{cwe_m.group(1)}" if cwe_m else "CWE-unknown"
                    bn = name.split('/')[-1].lower()
                    if '_good' in bn or 'good' in bn:
                        label = 0
                    elif '_bad' in bn or 'bad' in bn:
                        label = 1
                    else:
                        label = 1 if 'bad' in content.lower()[:200] else 0
                    lines = content.split('\n')
                    if len(lines) > 80:
                        lines = lines[:80]
                    lang_tag = "java" if language == "java" else "c_cpp"
                    samples.append({
                        "code": '\n'.join(lines), "label": label,
                        "language": lang_tag, "cwe": cwe,
                    })
                    if len(samples) >= max_samples:
                        break
        except Exception as e:
            print(f"  Juliet extract error ({language}): {e}")
        return samples

    java_zip = "/tmp/juliet_java.zip"
    if not os.path.exists(java_zip):
        print("  Downloading Juliet Java...")
        urllib.request.urlretrieve(juliet_java_url, java_zip)
        print(f"  Downloaded: {os.path.getsize(java_zip)/1e6:.1f} MB")
    java_samp = extract_juliet(java_zip, "java", 1000)
    if java_samp:
        add_samples(java_samp, "Juliet-Java")

    c_zip = "/tmp/juliet_c.zip"
    if not os.path.exists(c_zip):
        print("  Downloading Juliet C/C++...")
        urllib.request.urlretrieve(juliet_c_url, c_zip)
        print(f"  Downloaded: {os.path.getsize(c_zip)/1e6:.1f} MB")
    c_samp = extract_juliet(c_zip, "c", 1000)
    if c_samp:
        add_samples(c_samp, "Juliet-C")

    juliet_loaded = len(java_samp) > 0 or len(c_samp) > 0
except Exception as e:
    print(f"  Juliet download failed: {e}")

if not juliet_loaded:
    print("  Juliet not available")

# -------------------------------------------------------------------------
# 5. Synthetic Code Generators (fallback for any missing language)
# -------------------------------------------------------------------------
print("\n" + "="*60)
print("  5. Synthetic Code Generators")
print("="*60)

lang_counts = Counter(s["language"] for s in all_samples)
print(f"  Current coverage: {dict(lang_counts)}")

# --- Python templates ---
VULN_PY = [
    ('import sqlite3\ndef get_user(user_input):\n    conn = sqlite3.connect("app.db")\n    cursor = conn.cursor()\n    query = "SELECT * FROM users WHERE name = \'" + user_input + "\'"\n    cursor.execute(query)\n    results = cursor.fetchall()\n    conn.close()\n    return results', "CWE-89"),
    ('import os\nimport subprocess\ndef run_command(user_cmd):\n    command = "ls -la " + user_cmd\n    result = os.system(command)\n    output = subprocess.check_output(command, shell=True)\n    return output.decode("utf-8")', "CWE-78"),
    ('import os\ndef read_file(filename):\n    base_dir = "/var/www/uploads/"\n    filepath = base_dir + filename\n    with open(filepath, "r") as f:\n        content = f.read()\n    return content', "CWE-22"),
    ('import pickle\nimport base64\ndef load_data(serialized_input):\n    decoded = base64.b64decode(serialized_input)\n    data = pickle.loads(decoded)\n    return data', "CWE-502"),
    ('def calculate(expression):\n    user_expr = input("Enter expression: ")\n    result = eval(user_expr)\n    print(f"Result: {result}")\n    return result', "CWE-95"),
    ('from flask import Flask, request\napp = Flask(__name__)\n@app.route("/search")\ndef search():\n    query = request.args.get("q", "")\n    return "<html><body>Results for: " + query + "</body></html>"', "CWE-79"),
    ('import requests\ndef connect_api():\n    api_key = "sk-hardcoded-secret-key-12345"\n    password = "admin123"\n    headers = {"Authorization": f"Bearer {api_key}"}\n    resp = requests.get("https://api.example.com/data", headers=headers)\n    return resp.json()', "CWE-798"),
    ('import requests\nfrom flask import request as flask_request\ndef fetch_url():\n    url = flask_request.args.get("url")\n    response = requests.get(url)\n    return response.text', "CWE-918"),
    ('import random\ndef generate_token():\n    token = ""\n    chars = "abcdefghijklmnopqrstuvwxyz0123456789"\n    for i in range(32):\n        token += random.choice(chars)\n    return token', "CWE-330"),
    ('from lxml import etree\ndef parse_xml(xml_string):\n    parser = etree.XMLParser(resolve_entities=True)\n    tree = etree.fromstring(xml_string.encode(), parser)\n    return etree.tostring(tree)', "CWE-611"),
]
SAFE_PY = [
    ('import sqlite3\ndef get_user_safe(user_input):\n    conn = sqlite3.connect("app.db")\n    cursor = conn.cursor()\n    query = "SELECT * FROM users WHERE name = ?"\n    cursor.execute(query, (user_input,))\n    results = cursor.fetchall()\n    conn.close()\n    return results', "CWE-89"),
    ('import subprocess\nimport shlex\ndef run_command_safe(user_cmd):\n    allowed = {"ls", "pwd", "whoami", "date"}\n    if user_cmd not in allowed:\n        raise ValueError("Command not allowed")\n    result = subprocess.run([user_cmd], capture_output=True, text=True)\n    return result.stdout', "CWE-78"),
    ('import os\ndef read_file_safe(filename):\n    base_dir = "/var/www/uploads/"\n    filepath = os.path.join(base_dir, filename)\n    real_path = os.path.realpath(filepath)\n    if not real_path.startswith(os.path.realpath(base_dir)):\n        raise ValueError("Path traversal detected")\n    with open(real_path, "r") as f:\n        return f.read()', "CWE-22"),
    ('import json\ndef load_data_safe(serialized_input):\n    data = json.loads(serialized_input)\n    if not isinstance(data, dict):\n        raise ValueError("Expected dict")\n    return data', "CWE-502"),
    ('import ast\ndef calculate_safe(expression):\n    tree = ast.parse(expression, mode="eval")\n    for node in ast.walk(tree):\n        if isinstance(node, (ast.Call, ast.Attribute)):\n            raise ValueError("Unsafe expression")\n    result = ast.literal_eval(expression)\n    return result', "CWE-95"),
    ('from flask import Flask, request, escape\napp = Flask(__name__)\n@app.route("/search")\ndef search_safe():\n    query = request.args.get("q", "")\n    safe_query = escape(query)\n    return f"<html><body>Results for: {safe_query}</body></html>"', "CWE-79"),
    ('import os\ndef connect_api_safe():\n    api_key = os.environ.get("API_KEY")\n    if not api_key:\n        raise ValueError("API_KEY not configured")\n    headers = {"Authorization": f"Bearer {api_key}"}\n    return headers', "CWE-798"),
    ('import requests\nfrom urllib.parse import urlparse\ndef fetch_url_safe(url):\n    parsed = urlparse(url)\n    allowed_hosts = ["api.example.com", "cdn.example.com"]\n    if parsed.hostname not in allowed_hosts:\n        raise ValueError("Host not in allowlist")\n    response = requests.get(url, timeout=10)\n    return response.text', "CWE-918"),
    ('import secrets\ndef generate_token_safe():\n    token = secrets.token_urlsafe(32)\n    return token', "CWE-330"),
    ('from defusedxml import ElementTree\ndef parse_xml_safe(xml_string):\n    tree = ElementTree.fromstring(xml_string)\n    return ElementTree.tostring(tree)', "CWE-611"),
]

# --- JavaScript templates ---
VULN_JS = [
    ('function displaySearch(userInput) {\n    const results = document.getElementById("results");\n    results.innerHTML = "<h2>Search: " + userInput + "</h2>";\n    const container = document.createElement("div");\n    container.innerHTML = userInput;\n    document.body.appendChild(container);\n}', "CWE-79"),
    ('const express = require("express");\nconst app = express();\napp.get("/calc", (req, res) => {\n    const expr = req.query.expression;\n    const result = eval(expr);\n    res.json({ result: result });\n});', "CWE-95"),
    ('const mysql = require("mysql");\nfunction getUser(username) {\n    const conn = mysql.createConnection({host: "localhost", user: "root", database: "app"});\n    const query = "SELECT * FROM users WHERE name = \'" + username + "\'";\n    conn.query(query, (err, results) => {\n        return results;\n    });\n}', "CWE-89"),
    ('const fs = require("fs");\nconst path = require("path");\nfunction readFile(filename) {\n    const filepath = "./uploads/" + filename;\n    const content = fs.readFileSync(filepath, "utf8");\n    return content;\n}', "CWE-22"),
    ('function merge(target, source) {\n    for (let key in source) {\n        if (typeof source[key] === "object") {\n            target[key] = merge(target[key] || {}, source[key]);\n        } else {\n            target[key] = source[key];\n        }\n    }\n    return target;\n}', "CWE-1321"),
    ('const { exec } = require("child_process");\nfunction runLint(filename) {\n    const cmd = "eslint " + filename;\n    exec(cmd, (error, stdout, stderr) => {\n        console.log(stdout);\n    });\n}', "CWE-78"),
    ('const express = require("express");\nconst app = express();\napp.get("/redirect", (req, res) => {\n    const url = req.query.url;\n    res.redirect(url);\n});', "CWE-601"),
    ('const express = require("express");\nconst MongoClient = require("mongodb").MongoClient;\napp.post("/login", async (req, res) => {\n    const db = await MongoClient.connect("mongodb://localhost/app");\n    const user = await db.collection("users").findOne({\n        username: req.body.username,\n        password: req.body.password\n    });\n    res.json(user);\n});', "CWE-943"),
]
SAFE_JS = [
    ('function displaySearchSafe(userInput) {\n    const results = document.getElementById("results");\n    const textNode = document.createTextNode(userInput);\n    const heading = document.createElement("h2");\n    heading.appendChild(textNode);\n    results.innerHTML = "";\n    results.appendChild(heading);\n}', "CWE-79"),
    ('const express = require("express");\nconst app = express();\napp.get("/calc", (req, res) => {\n    const expr = req.query.expression;\n    const num = parseFloat(expr);\n    if (isNaN(num)) {\n        return res.status(400).json({ error: "Invalid number" });\n    }\n    res.json({ result: num * 2 });\n});', "CWE-95"),
    ('const mysql = require("mysql2");\nfunction getUserSafe(username) {\n    const conn = mysql.createConnection({host: "localhost", user: "root", database: "app"});\n    const query = "SELECT * FROM users WHERE name = ?";\n    conn.execute(query, [username], (err, results) => {\n        return results;\n    });\n}', "CWE-89"),
    ('const fs = require("fs");\nconst path = require("path");\nfunction readFileSafe(filename) {\n    const baseDir = path.resolve("./uploads");\n    const filepath = path.resolve(path.join(baseDir, filename));\n    if (!filepath.startsWith(baseDir)) {\n        throw new Error("Path traversal detected");\n    }\n    return fs.readFileSync(filepath, "utf8");\n}', "CWE-22"),
    ('function mergeSafe(target, source) {\n    const BLOCKED = ["__proto__", "constructor", "prototype"];\n    for (let key in source) {\n        if (BLOCKED.includes(key)) continue;\n        if (!source.hasOwnProperty(key)) continue;\n        if (typeof source[key] === "object" && source[key] !== null) {\n            target[key] = mergeSafe(target[key] || {}, source[key]);\n        } else {\n            target[key] = source[key];\n        }\n    }\n    return target;\n}', "CWE-1321"),
    ('const { execFile } = require("child_process");\nfunction runLintSafe(filename) {\n    const safePattern = /^[a-zA-Z0-9_.-]+$/;\n    if (!safePattern.test(filename)) {\n        throw new Error("Invalid filename");\n    }\n    execFile("eslint", [filename], (error, stdout) => {\n        console.log(stdout);\n    });\n}', "CWE-78"),
    ('const express = require("express");\nconst app = express();\napp.get("/redirect", (req, res) => {\n    const url = req.query.url;\n    const allowed = ["https://example.com", "https://app.example.com"];\n    if (!allowed.some(a => url.startsWith(a))) {\n        return res.status(400).send("Redirect not allowed");\n    }\n    res.redirect(url);\n});', "CWE-601"),
    ('const express = require("express");\nconst bcrypt = require("bcrypt");\napp.post("/login", async (req, res) => {\n    const { username, password } = req.body;\n    if (typeof username !== "string" || typeof password !== "string") {\n        return res.status(400).json({ error: "Invalid input" });\n    }\n    const user = await db.collection("users").findOne({ username: String(username) });\n    if (user && await bcrypt.compare(password, user.passwordHash)) {\n        res.json({ success: true });\n    } else {\n        res.status(401).json({ error: "Invalid credentials" });\n    }\n});', "CWE-943"),
]

# --- Go templates ---
VULN_GO = [
    ('package main\nimport (\n    "database/sql"\n    "fmt"\n    "net/http"\n    _ "github.com/lib/pq"\n)\nfunc getUser(w http.ResponseWriter, r *http.Request) {\n    name := r.FormValue("name")\n    query := fmt.Sprintf("SELECT * FROM users WHERE name = \'%s\'", name)\n    db, _ := sql.Open("postgres", "connstring")\n    rows, _ := db.Query(query)\n    defer rows.Close()\n    fmt.Fprintf(w, "Results: %v", rows)\n}', "CWE-89"),
    ('package main\nimport (\n    "net/http"\n    "os/exec"\n)\nfunc handler(w http.ResponseWriter, r *http.Request) {\n    host := r.FormValue("host")\n    cmd := exec.Command("ping", "-c", "4", host)\n    output, _ := cmd.CombinedOutput()\n    w.Write(output)\n}', "CWE-78"),
    ('package main\nimport (\n    "io/ioutil"\n    "net/http"\n)\nfunc readFile(w http.ResponseWriter, r *http.Request) {\n    filename := r.URL.Query().Get("file")\n    filepath := "/var/www/uploads/" + filename\n    data, err := ioutil.ReadFile(filepath)\n    if err != nil {\n        http.Error(w, "File not found", 404)\n        return\n    }\n    w.Write(data)\n}', "CWE-22"),
    ('package main\nimport (\n    "io/ioutil"\n    "net/http"\n)\nfunc fetchURL(w http.ResponseWriter, r *http.Request) {\n    url := r.FormValue("url")\n    resp, err := http.Get(url)\n    if err != nil {\n        http.Error(w, "Fetch failed", 500)\n        return\n    }\n    defer resp.Body.Close()\n    body, _ := ioutil.ReadAll(resp.Body)\n    w.Write(body)\n}', "CWE-918"),
    ('package main\nimport (\n    "encoding/gob"\n    "bytes"\n    "net/http"\n    "io/ioutil"\n)\ntype UserData struct {\n    Name string\n    Role string\n}\nfunc deserialize(w http.ResponseWriter, r *http.Request) {\n    body, _ := ioutil.ReadAll(r.Body)\n    buf := bytes.NewBuffer(body)\n    dec := gob.NewDecoder(buf)\n    var data UserData\n    dec.Decode(&data)\n}', "CWE-502"),
]
SAFE_GO = [
    ('package main\nimport (\n    "database/sql"\n    "net/http"\n    "encoding/json"\n    _ "github.com/lib/pq"\n)\nfunc getUserSafe(w http.ResponseWriter, r *http.Request) {\n    name := r.FormValue("name")\n    db, _ := sql.Open("postgres", "connstring")\n    defer db.Close()\n    rows, err := db.Query("SELECT id, name FROM users WHERE name = $1", name)\n    if err != nil {\n        http.Error(w, "Query error", 500)\n        return\n    }\n    defer rows.Close()\n    json.NewEncoder(w).Encode(rows)\n}', "CWE-89"),
    ('package main\nimport (\n    "net"\n    "net/http"\n    "os/exec"\n    "regexp"\n)\nfunc handlerSafe(w http.ResponseWriter, r *http.Request) {\n    host := r.FormValue("host")\n    ipPattern := regexp.MustCompile(`^[0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3}$`)\n    if !ipPattern.MatchString(host) {\n        http.Error(w, "Invalid IP", 400)\n        return\n    }\n    if net.ParseIP(host) == nil {\n        http.Error(w, "Invalid IP", 400)\n        return\n    }\n    cmd := exec.Command("ping", "-c", "4", host)\n    output, _ := cmd.CombinedOutput()\n    w.Write(output)\n}', "CWE-78"),
    ('package main\nimport (\n    "io/ioutil"\n    "net/http"\n    "path/filepath"\n    "strings"\n)\nfunc readFileSafe(w http.ResponseWriter, r *http.Request) {\n    filename := r.URL.Query().Get("file")\n    baseDir := "/var/www/uploads/"\n    fullPath := filepath.Join(baseDir, filename)\n    absPath, _ := filepath.Abs(fullPath)\n    absBase, _ := filepath.Abs(baseDir)\n    if !strings.HasPrefix(absPath, absBase) {\n        http.Error(w, "Forbidden", 403)\n        return\n    }\n    data, err := ioutil.ReadFile(absPath)\n    if err != nil {\n        http.Error(w, "Not found", 404)\n        return\n    }\n    w.Write(data)\n}', "CWE-22"),
    ('package main\nimport (\n    "io/ioutil"\n    "net/http"\n    "net/url"\n)\nfunc fetchURLSafe(w http.ResponseWriter, r *http.Request) {\n    rawURL := r.FormValue("url")\n    parsed, err := url.Parse(rawURL)\n    if err != nil {\n        http.Error(w, "Invalid URL", 400)\n        return\n    }\n    allowedHosts := map[string]bool{"api.example.com": true}\n    if !allowedHosts[parsed.Host] {\n        http.Error(w, "Host not allowed", 403)\n        return\n    }\n    resp, _ := http.Get(parsed.String())\n    defer resp.Body.Close()\n    body, _ := ioutil.ReadAll(resp.Body)\n    w.Write(body)\n}', "CWE-918"),
    ('package main\nimport (\n    "encoding/json"\n    "net/http"\n    "io/ioutil"\n)\ntype UserData struct {\n    Name string `json:"name"`\n    Role string `json:"role"`\n}\nfunc deserializeSafe(w http.ResponseWriter, r *http.Request) {\n    body, _ := ioutil.ReadAll(r.Body)\n    var data UserData\n    if err := json.Unmarshal(body, &data); err != nil {\n        http.Error(w, "Invalid JSON", 400)\n        return\n    }\n    allowedRoles := map[string]bool{"user": true, "viewer": true}\n    if !allowedRoles[data.Role] {\n        data.Role = "user"\n    }\n}', "CWE-502"),
]

# --- C/C++ templates ---
VULN_C = [
    ('#include <stdio.h>\n#include <string.h>\nvoid process_input(char *user_input) {\n    char buffer[64];\n    strcpy(buffer, user_input);\n    printf("Input: %s\\n", buffer);\n}', "CWE-120"),
    ('#include <stdlib.h>\n#include <string.h>\nvoid copy_data(const char *src) {\n    char *dst = (char *)malloc(32);\n    memcpy(dst, src, strlen(src));\n    printf("Copied: %s\\n", dst);\n}', "CWE-122"),
    ('#include <stdio.h>\n#include <stdlib.h>\nvoid read_file(const char *filename) {\n    char path[256];\n    sprintf(path, "/var/data/%s", filename);\n    FILE *f = fopen(path, "r");\n    char buf[1024];\n    fread(buf, 1, sizeof(buf), f);\n    fclose(f);\n}', "CWE-22"),
    ('#include <stdlib.h>\nint compute(int a, int b) {\n    int result = a * b;\n    char *buf = (char *)malloc(result);\n    if (!buf) return -1;\n    memset(buf, 0, result);\n    free(buf);\n    return 0;\n}', "CWE-190"),
    ('#include <stdio.h>\nvoid format_log(const char *user_msg) {\n    printf(user_msg);\n    fprintf(stderr, user_msg);\n}', "CWE-134"),
]
SAFE_C = [
    ('#include <stdio.h>\n#include <string.h>\nvoid process_input_safe(const char *user_input) {\n    char buffer[64];\n    strncpy(buffer, user_input, sizeof(buffer) - 1);\n    buffer[sizeof(buffer) - 1] = \'\\0\';\n    printf("Input: %s\\n", buffer);\n}', "CWE-120"),
    ('#include <stdlib.h>\n#include <string.h>\nvoid copy_data_safe(const char *src) {\n    size_t len = strlen(src);\n    if (len > 1024) return;\n    char *dst = (char *)malloc(len + 1);\n    if (!dst) return;\n    memcpy(dst, src, len);\n    dst[len] = \'\\0\';\n    free(dst);\n}', "CWE-122"),
    ('#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nvoid read_file_safe(const char *filename) {\n    if (strstr(filename, "..") != NULL) return;\n    char path[256];\n    snprintf(path, sizeof(path), "/var/data/%s", filename);\n    FILE *f = fopen(path, "r");\n    if (!f) return;\n    char buf[1024];\n    size_t n = fread(buf, 1, sizeof(buf) - 1, f);\n    buf[n] = \'\\0\';\n    fclose(f);\n}', "CWE-22"),
    ('#include <stdlib.h>\n#include <limits.h>\nint compute_safe(int a, int b) {\n    if (a > 0 && b > 0 && a > INT_MAX / b) return -1;\n    int result = a * b;\n    if (result <= 0 || result > 1048576) return -1;\n    char *buf = (char *)malloc(result);\n    if (!buf) return -1;\n    memset(buf, 0, result);\n    free(buf);\n    return 0;\n}', "CWE-190"),
    ('#include <stdio.h>\nvoid format_log_safe(const char *user_msg) {\n    printf("%s", user_msg);\n    fprintf(stderr, "%s", user_msg);\n}', "CWE-134"),
]

# --- Java templates ---
VULN_JAVA = [
    ('import java.sql.*;\npublic class UserDAO {\n    public ResultSet getUser(String username) throws SQLException {\n        Connection conn = DriverManager.getConnection("jdbc:mysql://localhost/app");\n        Statement stmt = conn.createStatement();\n        String query = "SELECT * FROM users WHERE name = \'" + username + "\'";\n        return stmt.executeQuery(query);\n    }\n}', "CWE-89"),
    ('import java.io.*;\npublic class CommandRunner {\n    public String runCommand(String cmd) throws IOException {\n        Runtime rt = Runtime.getRuntime();\n        Process proc = rt.exec("sh -c " + cmd);\n        BufferedReader reader = new BufferedReader(new InputStreamReader(proc.getInputStream()));\n        StringBuilder output = new StringBuilder();\n        String line;\n        while ((line = reader.readLine()) != null) {\n            output.append(line);\n        }\n        return output.toString();\n    }\n}', "CWE-78"),
    ('import javax.servlet.http.*;\nimport java.io.*;\npublic class XSSServlet extends HttpServlet {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response)\n            throws IOException {\n        String name = request.getParameter("name");\n        response.getWriter().println("<html><body>Hello " + name + "</body></html>");\n    }\n}', "CWE-79"),
]
SAFE_JAVA = [
    ('import java.sql.*;\npublic class UserDAOSafe {\n    public ResultSet getUserSafe(String username) throws SQLException {\n        Connection conn = DriverManager.getConnection("jdbc:mysql://localhost/app");\n        PreparedStatement stmt = conn.prepareStatement("SELECT * FROM users WHERE name = ?");\n        stmt.setString(1, username);\n        return stmt.executeQuery();\n    }\n}', "CWE-89"),
    ('import java.io.*;\nimport java.util.*;\npublic class CommandRunnerSafe {\n    private static final Set<String> ALLOWED = Set.of("ls", "pwd", "date");\n    public String runCommandSafe(String cmd) throws IOException {\n        if (!ALLOWED.contains(cmd)) {\n            throw new SecurityException("Command not allowed: " + cmd);\n        }\n        ProcessBuilder pb = new ProcessBuilder(cmd);\n        pb.redirectErrorStream(true);\n        Process proc = pb.start();\n        BufferedReader reader = new BufferedReader(new InputStreamReader(proc.getInputStream()));\n        StringBuilder output = new StringBuilder();\n        String line;\n        while ((line = reader.readLine()) != null) {\n            output.append(line);\n        }\n        return output.toString();\n    }\n}', "CWE-78"),
    ('import javax.servlet.http.*;\nimport java.io.*;\npublic class SafeServlet extends HttpServlet {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response)\n            throws IOException {\n        String name = request.getParameter("name");\n        String safeName = name.replace("<", "&lt;").replace(">", "&gt;");\n        response.getWriter().println("<html><body>Hello " + safeName + "</body></html>");\n    }\n}', "CWE-79"),
]

def variate_code(template, n_variants):
    """Create n variants of a code template with minor modifications."""
    variants = [template]
    var_names = ["data", "value", "item", "output", "response", "temp",
                 "payload", "content", "info", "record", "entry", "obj", "msg"]
    for i in range(n_variants - 1):
        c = template
        vn = random.choice(var_names)
        c = c.replace("result", vn, 1)
        comments = ["# Processing input", "# Handle request", "# Main logic",
                     "// Process data", "// Execute operation"]
        if random.random() > 0.5:
            lines = c.split('\n')
            pos = min(3, len(lines))
            lines.insert(pos, "    " + random.choice(comments))
            c = '\n'.join(lines)
        variants.append(c)
    return variants

def generate_synthetic(vuln_t, safe_t, language, n_vuln=200, n_safe=200):
    samples = []
    vpp = max(1, n_vuln // len(vuln_t))
    for tpl, cwe in vuln_t:
        for v in variate_code(tpl, vpp):
            samples.append({"code": v, "label": 1, "language": language, "cwe": cwe})
            if sum(1 for s in samples if s["label"] == 1) >= n_vuln:
                break
        if sum(1 for s in samples if s["label"] == 1) >= n_vuln:
            break
    spp = max(1, n_safe // len(safe_t))
    for tpl, cwe in safe_t:
        for v in variate_code(tpl, spp):
            samples.append({"code": v, "label": 0, "language": language, "cwe": cwe})
            if sum(1 for s in samples if s["label"] == 0) >= n_safe:
                break
        if sum(1 for s in samples if s["label"] == 0) >= n_safe:
            break
    return samples

SYNTH_MAP = {
    "python": (VULN_PY, SAFE_PY, 200, 200),
    "javascript": (VULN_JS, SAFE_JS, 200, 200),
    "go": (VULN_GO, SAFE_GO, 150, 150),
    "c_cpp": (VULN_C, SAFE_C, 200, 200),
    "java": (VULN_JAVA, SAFE_JAVA, 200, 200),
}

for lang, (vt, st, nv, ns) in SYNTH_MAP.items():
    existing = lang_counts.get(lang, 0)
    if existing < 100:
        print(f"\n  Generating synthetic {lang} ({nv} vuln + {ns} safe)...")
        synth = generate_synthetic(vt, st, lang, nv, ns)
        add_samples(synth, f"synthetic-{lang}")
    else:
        print(f"\n  {lang}: {existing} real samples available, skipping synthetic")

print(f"\n{'='*60}")
print(f"  Total raw samples: {len(all_samples)}")
lang_final = Counter(s["language"] for s in all_samples)
for lang, count in sorted(lang_final.items()):
    labels = Counter(s["label"] for s in all_samples if s["language"] == lang)
    print(f"    {lang}: {count} ({dict(labels)})")
print(f"{'='*60}")

  1. CVEfixes Dataset
  Trying Kaggle API download...
  Loading CSV from /tmp/cvefixes/CVEFixes.csv...
  Columns: ['code', 'language', 'safety'], Shape: (16959859, 3)
  [CVEfixes] Added 5000 samples: {0: 2599, 1: 2401}, langs={'c_cpp': 3217, 'java': 429, 'python': 586, 'javascript': 599, 'go': 169}

  2. DiverseVul Dataset (C/C++)
  Trying HuggingFace bstee615/diversevul (streaming)...


README.md: 0.00B [00:00, ?B/s]

  [DiverseVul] Added 2933 samples: {1: 176, 0: 2757}, langs={'c_cpp': 2933}

  3. Devign Dataset (C/C++)
  Trying HuggingFace DetectVul/devign (streaming)...


README.md: 0.00B [00:00, ?B/s]

  [Devign] Added 2000 samples: {0: 1034, 1: 966}, langs={'c_cpp': 2000}

  4. Juliet Test Suite (Java + C/C++)
  Downloaded: 76.8 MB
  [Juliet-Java] Added 1000 samples: {0: 983, 1: 17}, langs={'java': 1000}
  Downloaded: 153.0 MB
  [Juliet-C] Added 1000 samples: {0: 995, 1: 5}, langs={'c_cpp': 1000}

  5. Synthetic Code Generators
  Current coverage: {'c_cpp': 9150, 'java': 1429, 'python': 586, 'javascript': 599, 'go': 169}

  python: 586 real samples available, skipping synthetic

  javascript: 599 real samples available, skipping synthetic

  go: 169 real samples available, skipping synthetic

  c_cpp: 9150 real samples available, skipping synthetic

  java: 1429 real samples available, skipping synthetic

  Total raw samples: 11933
    c_cpp: 9150 ({0: 6452, 1: 2698})
    go: 169 ({0: 92, 1: 77})
    java: 1429 ({0: 1207, 1: 222})
    javascript: 599 ({1: 289, 0: 310})
    python: 586 ({1: 279, 0: 307})


## Cell 4: Dataset Balancing & Cleaning

In [4]:
# ============================================================================
# Cell 4: Dataset Balancing & Cleaning
# ============================================================================
import hashlib
from collections import Counter

print("="*60)
print("  Dataset Balancing & Cleaning")
print("="*60)

# 1. Remove duplicates by code hash
print("\n1. Removing duplicates...")
seen_hashes = set()
unique_samples = []
for s in all_samples:
    h = hashlib.md5(s["code"].strip().encode()).hexdigest()
    if h not in seen_hashes:
        seen_hashes.add(h)
        unique_samples.append(s)
print(f"   Before: {len(all_samples)}, After dedup: {len(unique_samples)}")

# 2. Remove samples with < 3 lines
print("\n2. Removing short samples (< 3 lines)...")
filtered = [s for s in unique_samples if len(s["code"].strip().split('\n')) >= 3]
print(f"   Before: {len(unique_samples)}, After filter: {len(filtered)}")

# 3. Balance per language (max 2:1 vuln:safe)
print(f"\n3. Balancing to max {CONFIG['target_balance_ratio']}:1 ratio...")
balanced = []
for lang in CONFIG["language_ids"]:
    lang_s = [s for s in filtered if s["language"] == lang]
    vuln = [s for s in lang_s if s["label"] == 1]
    safe = [s for s in lang_s if s["label"] == 0]
    n_vuln, n_safe = len(vuln), len(safe)
    if n_safe == 0:
        print(f"   {lang}: {n_vuln} vuln, 0 safe -- SKIPPING")
        continue
    ratio = n_vuln / max(n_safe, 1)
    if ratio > CONFIG["target_balance_ratio"]:
        max_vuln = int(n_safe * CONFIG["target_balance_ratio"])
        random.shuffle(vuln)
        vuln = vuln[:max_vuln]
        print(f"   {lang}: {n_vuln} -> {len(vuln)} vuln (downsampled), {n_safe} safe")
    else:
        print(f"   {lang}: {n_vuln} vuln, {n_safe} safe (ratio {ratio:.1f}:1, OK)")
    balanced.extend(vuln)
    balanced.extend(safe)

all_samples = balanced
random.shuffle(all_samples)

# Final stats
total_vuln = sum(1 for s in all_samples if s["label"] == 1)
total_safe = sum(1 for s in all_samples if s["label"] == 0)

print(f"\n{'='*60}")
print(f"  Final dataset: {len(all_samples)} samples")
print(f"{'='*60}")
print(f"\n  Per-language:")
for lang in CONFIG["language_ids"]:
    ls = [s for s in all_samples if s["language"] == lang]
    lbl = Counter(s["label"] for s in ls)
    src = Counter(s["source"] for s in ls)
    print(f"    {lang}: {len(ls)} (vuln={lbl.get(1,0)}, safe={lbl.get(0,0)}) | {dict(src)}")
print(f"\n  Overall: {total_vuln} vuln + {total_safe} safe = {len(all_samples)}")
print(f"  Ratio: {total_vuln/max(total_safe,1):.2f}:1")

  Dataset Balancing & Cleaning

1. Removing duplicates...
   Before: 11933, After dedup: 11339

2. Removing short samples (< 3 lines)...
   Before: 11339, After filter: 11339

3. Balancing to max 2.0:1 ratio...
   python: 251 vuln, 290 safe (ratio 0.9:1, OK)
   javascript: 259 vuln, 277 safe (ratio 0.9:1, OK)
   java: 199 vuln, 1176 safe (ratio 0.2:1, OK)
   c_cpp: 2482 vuln, 6242 safe (ratio 0.4:1, OK)
   go: 74 vuln, 89 safe (ratio 0.8:1, OK)

  Final dataset: 11339 samples

  Per-language:
    python: 541 (vuln=251, safe=290) | {'CVEfixes': 541}
    javascript: 536 (vuln=259, safe=277) | {'CVEfixes': 536}
    java: 1375 (vuln=199, safe=1176) | {'CVEfixes': 387, 'Juliet-Java': 988}
    c_cpp: 8724 (vuln=2482, safe=6242) | {'Juliet-C': 1000, 'DiverseVul': 2933, 'CVEfixes': 2791, 'Devign': 2000}
    go: 163 (vuln=74, safe=89) | {'CVEfixes': 163}

  Overall: 3265 vuln + 8074 safe = 11339
  Ratio: 0.40:1


## Cell 5: Exploratory Data Analysis

In [5]:
# ============================================================================
# Cell 5: Exploratory Data Analysis
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Language distribution
ax1 = axes[0, 0]
lang_names = list(CONFIG["language_ids"].keys())
lang_vals = [sum(1 for s in all_samples if s["language"] == l) for l in lang_names]
colors = ['#4dabf7', '#ffa94d', '#51cf66', '#ff6b6b', '#cc5de8']
bars = ax1.bar(lang_names, lang_vals, color=colors)
ax1.set_title("Language Distribution", fontweight='bold')
ax1.set_ylabel("Count")
for bar, val in zip(bars, lang_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontweight='bold')

# 2. CWE distribution (top 15)
ax2 = axes[0, 1]
cwe_counts = Counter(s["cwe"] for s in all_samples).most_common(15)
cwe_n, cwe_v = [c[0] for c in cwe_counts], [c[1] for c in cwe_counts]
ax2.barh(cwe_n[::-1], cwe_v[::-1], color='#748ffc')
ax2.set_title("Top 15 CWEs", fontweight='bold')
ax2.set_xlabel("Count")

# 3. Class balance per language
ax3 = axes[1, 0]
x_pos = np.arange(len(lang_names))
vuln_pl = [sum(1 for s in all_samples if s["language"]==l and s["label"]==1) for l in lang_names]
safe_pl = [sum(1 for s in all_samples if s["language"]==l and s["label"]==0) for l in lang_names]
w = 0.35
ax3.bar(x_pos - w/2, vuln_pl, w, label='Vulnerable', color='#ff6b6b')
ax3.bar(x_pos + w/2, safe_pl, w, label='Safe', color='#51cf66')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(lang_names)
ax3.set_title("Class Balance per Language", fontweight='bold')
ax3.legend()

# 4. Code length histogram
ax4 = axes[1, 1]
lengths = [len(s["code"].split('\n')) for s in all_samples]
ax4.hist(lengths, bins=50, color='#845ef7', alpha=0.7, edgecolor='black')
ax4.axvline(np.median(lengths), color='red', linestyle='--',
            label=f'Median: {np.median(lengths):.0f}')
ax4.set_title("Code Length Distribution", fontweight='bold')
ax4.set_xlabel("Lines")
ax4.legend()

plt.suptitle("SEC-C V2 Dataset Overview", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "eda_overview.png"), dpi=150)
plt.show()

# Summary table
print("\n" + "="*70)
print(f"{'Language':<15} {'Total':>7} {'Vuln':>7} {'Safe':>7} {'Ratio':>8}")
print("-"*70)
for i, l in enumerate(lang_names):
    nv, ns = vuln_pl[i], safe_pl[i]
    r = f"{nv/max(ns,1):.1f}:1"
    print(f"{l:<15} {nv+ns:>7} {nv:>7} {ns:>7} {r:>8}")
print("-"*70)
print(f"{'TOTAL':<15} {len(all_samples):>7} {total_vuln:>7} {total_safe:>7} {total_vuln/max(total_safe,1):.1f}:1")
print("="*70)


Language          Total    Vuln    Safe    Ratio
----------------------------------------------------------------------
python              541     251     290    0.9:1
javascript          536     259     277    0.9:1
java               1375     199    1176    0.2:1
c_cpp              8724    2482    6242    0.4:1
go                  163      74      89    0.8:1
----------------------------------------------------------------------
TOTAL             11339    3265    8074 0.4:1


## Cell 6: Build Code Graphs (Improved Regex Fallback)

In [6]:
# ============================================================================
# Cell 6: Build Code Graphs (AST + CFG + DDG via Regex)
# ============================================================================
import networkx as nx
import re
from collections import Counter

BRANCH_KW = {
    "python": re.compile(r'^\s*(if|elif|else|for|while|try|except|finally|with|def|class|async)\b'),
    "javascript": re.compile(r'^\s*(if|else|for|while|do|switch|case|try|catch|finally|function|async|class)\b'),
    "java": re.compile(r'^\s*(if|else|for|while|do|switch|case|try|catch|finally|class|public|private|protected)\b'),
    "c_cpp": re.compile(r'^\s*(if|else|for|while|do|switch|case|goto|struct|typedef)\b'),
    "go": re.compile(r'^\s*(if|else|for|switch|case|select|func|go|defer|type)\b'),
}
RETURN_KW = re.compile(r'^\s*(return|raise|throw|panic|break|continue|goto)\b')
VAR_DEF = re.compile(r'(\b[a-zA-Z_]\w*)\s*(?:=|:=|<-)')
SKIP_VARS = {'if', 'for', 'while', 'return', 'else', 'self', 'this', 'true', 'false',
             'nil', 'null', 'None', 'var', 'let', 'const', 'int', 'string', 'bool'}

def get_indent(line):
    stripped = line.lstrip()
    return len(line) - len(stripped) if stripped else 0

def build_code_graph(code: str, language: str) -> nx.DiGraph:
    """Build code graph with AST + CFG + DDG edges from regex analysis."""
    G = nx.DiGraph()
    lines = code.split('\n')
    non_empty = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped and not stripped.startswith(('#', '//', '/*', '*', '*/')):
            non_empty.append((i, line, stripped))
    if not non_empty:
        G.add_node(0, text="empty", indent=0)
        return G
    for idx, (orig, line, stripped) in enumerate(non_empty):
        G.add_node(idx, text=stripped, indent=get_indent(line), orig_line=orig)

    # AST edges (indentation parent-child)
    indent_stack = []
    for idx, (_, line, _) in enumerate(non_empty):
        indent = get_indent(line)
        while indent_stack and indent_stack[-1][1] >= indent:
            indent_stack.pop()
        if indent_stack:
            G.add_edge(indent_stack[-1][0], idx, edge_type='ast')
        indent_stack.append((idx, indent))

    # CFG edges
    branch_re = BRANCH_KW.get(language, BRANCH_KW["python"])
    for idx in range(len(non_empty) - 1):
        _, line, stripped = non_empty[idx]
        G.add_edge(idx, idx + 1, edge_type='cfg')
        if branch_re.match(stripped):
            cur_indent = get_indent(line)
            for j in range(idx + 2, len(non_empty)):
                if get_indent(non_empty[j][1]) <= cur_indent:
                    G.add_edge(idx, j, edge_type='cfg_branch')
                    break
        if RETURN_KW.match(stripped):
            G.add_edge(idx, len(non_empty) - 1, edge_type='cfg_return')

    # DDG edges (variable def-use)
    var_defs = {}
    for idx, (_, _, stripped) in enumerate(non_empty):
        for m in VAR_DEF.finditer(stripped):
            vn = m.group(1)
            if vn not in SKIP_VARS:
                var_defs.setdefault(vn, []).append(idx)
    for idx, (_, _, stripped) in enumerate(non_empty):
        words = set(re.findall(r'\b([a-zA-Z_]\w*)\b', stripped))
        for vn in words:
            if vn in var_defs:
                defs_before = [d for d in var_defs[vn] if d < idx]
                if defs_before and defs_before[-1] != idx:
                    G.add_edge(defs_before[-1], idx, edge_type='ddg')
    return G

# Test
test_code = 'def process(request):\n    name = request.args.get("name")\n    if not name:\n        return "Error"\n    query = "SELECT * FROM users WHERE name = \'" + name + "\'"\n    result = db.execute(query)\n    return result'
G_test = build_code_graph(test_code, "python")
et = Counter(d.get('edge_type','?') for _,_,d in G_test.edges(data=True))
print(f"Test graph: {G_test.number_of_nodes()} nodes, {G_test.number_of_edges()} edges")
print(f"  Edge types: {dict(et)}")
print("  Graph construction ready.")

# ── Checkpoint: save raw samples so Cell 7 (embedding) can resume ─────────────
import json as _json_ckpt
_raw_ckpt = OUTPUT_DIR / "raw_samples_v2.json"
try:
    with open(str(_raw_ckpt), "w", encoding="utf-8") as _f:
        _json_ckpt.dump(all_samples, _f)
    print(f"\n  [CKPT] Saved {len(all_samples)} raw samples -> {_raw_ckpt}")
except Exception as _e:
    print(f"  [CKPT] Could not save raw samples: {_e}")


Test graph: 7 nodes, 13 edges
  Edge types: {'cfg': 3, 'ast': 4, 'ddg': 4, 'cfg_branch': 1, 'cfg_return': 1}
  Graph construction ready.

  [CKPT] Saved 11339 raw samples -> /kaggle/working/raw_samples_v2.json


## Cell 7: Build PyG Dataset with GraphCodeBERT Embeddings

In [7]:
# ============================================================================
# Cell 7: Build PyG Dataset with GraphCodeBERT Embeddings
# ============================================================================
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading GraphCodeBERT...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"])
gcb_model = AutoModel.from_pretrained(CONFIG["embedding_model"]).to(device)
gcb_model.eval()
print(f"  Loaded on {device} (pre-trained: Python, JS, Java, Go, PHP, Ruby)")

def is_sink_node(text):
    t = text.lower()
    return any(p in t for p in SINK_PATTERNS)

def is_source_node(text):
    t = text.lower()
    return any(p in t for p in SOURCE_PATTERNS)

def embed_nodes_batch(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, truncation=True,
                           max_length=CONFIG["max_tokens_per_node"],
                           return_tensors="pt").to(device)
        with torch.no_grad():
            out = gcb_model(**tokens)
            cls_emb = out.last_hidden_state[:, 0, :]
            embeddings.append(cls_emb.cpu())
    return torch.cat(embeddings, dim=0) if embeddings else torch.empty(0, 768)

def build_pyg_data(sample, max_nodes=300):
    code_text = sample["code"]
    label = sample["label"]
    language = sample["language"]
    G = build_code_graph(code_text, language)
    nodes = list(G.nodes(data=True))
    if len(nodes) > max_nodes:
        nodes = nodes[:max_nodes]
        keep = set(n[0] for n in nodes)
        G = G.subgraph(keep).copy()
        mapping = {old: new for new, old in enumerate(sorted(G.nodes()))}
        G = nx.relabel_nodes(G, mapping)
        nodes = list(G.nodes(data=True))
    if not nodes:
        return None
    node_texts = [d.get("text", "empty") for _, d in nodes]
    gcb_emb = embed_nodes_batch(node_texts, CONFIG["embedding_batch_size"])
    if gcb_emb.shape[0] == 0:
        return None
    n = gcb_emb.shape[0]
    in_deg = np.array([G.in_degree(i) for i in range(n)], dtype=np.float32)
    out_deg = np.array([G.out_degree(i) for i in range(n)], dtype=np.float32)
    in_norm = in_deg / max(in_deg.max(), 1.0)
    out_norm = out_deg / max(out_deg.max(), 1.0)
    sink = np.array([1.0 if is_sink_node(node_texts[i]) else 0.0 for i in range(n)], dtype=np.float32)
    source = np.array([1.0 if is_source_node(node_texts[i]) else 0.0 for i in range(n)], dtype=np.float32)
    try:
        if n > 0 and G.number_of_edges() > 0:
            lengths = nx.single_source_shortest_path_length(G, 0)
            depth = np.array([lengths.get(i, 0) for i in range(n)], dtype=np.float32)
        else:
            depth = np.zeros(n, dtype=np.float32)
    except Exception:
        depth = np.zeros(n, dtype=np.float32)
    depth_norm = depth / max(depth.max(), 1.0)
    lang_id = CONFIG["language_ids"].get(language, 0.5)
    lang_feat = np.full(n, lang_id, dtype=np.float32)
    structural = np.stack([in_norm, out_norm, sink, source, depth_norm, lang_feat], axis=1)
    x = torch.cat([gcb_emb, torch.tensor(structural, dtype=torch.float32)], dim=1)
    edges = list(G.edges())
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    y = torch.tensor(label, dtype=torch.long)
    data = Data(x=x, edge_index=edge_index, y=y)
    data.language = language
    data.cwe = sample.get("cwe", "unknown")
    data.source_ds = sample.get("source", "unknown")
    return data

# ── Load checkpoint if available (saves 3+ hours of re-embedding) ─────────────
_pyg_ckpt  = OUTPUT_DIR / "pyg_dataset_v2_partial.pt"
_pyg_done  = OUTPUT_DIR / "pyg_dataset_v2.pt"

if _pyg_done.exists():
    pyg_dataset = torch.load(str(_pyg_done), weights_only=False)
    print(f"[CKPT] Loaded complete PyG dataset: {len(pyg_dataset)} graphs from {_pyg_done}")
else:
    # Resume from partial checkpoint if it exists
    if _pyg_ckpt.exists():
        pyg_dataset = torch.load(str(_pyg_ckpt), weights_only=False)
        _start_idx = len(pyg_dataset)
        print(f"[CKPT] Resuming from {_start_idx} already-embedded graphs")
    else:
        pyg_dataset = []
        _start_idx = 0

    print(f"\nBuilding PyG dataset ({len(all_samples)} samples, starting at {_start_idx})...")
    print(f"  Features: {CONFIG['embedding_dim']} + {CONFIG['node_feature_dim']} = {CONFIG['input_dim']}")

    failed = 0
    _SAVE_EVERY = 500  # periodic partial checkpoint every 500 graphs
    for i, sample in enumerate(tqdm(all_samples[_start_idx:], desc="Building graphs",
                                    initial=_start_idx, total=len(all_samples))):
        try:
            data = build_pyg_data(sample, CONFIG["max_nodes"])
            if data is not None:
                pyg_dataset.append(data)
            else:
                failed += 1
        except Exception as e:
            failed += 1
            if i < 3:
                print(f"  Error {_start_idx+i}: {e}")

        # Periodic save so a crash doesn't lose everything
        if len(pyg_dataset) > 0 and len(pyg_dataset) % _SAVE_EVERY == 0:
            try:
                torch.save(pyg_dataset, str(_pyg_ckpt))
            except Exception:
                pass  # don't let a save failure abort embedding

    print(f"\n  Built {len(pyg_dataset)} graphs, {failed} failed")

del gcb_model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  Freed GraphCodeBERT from GPU")

if pyg_dataset:
    s0 = pyg_dataset[0]
    print(f"  Sample: {s0.x.shape[0]} nodes, {s0.edge_index.shape[1]} edges, dim={s0.x.shape[1]}")
    assert s0.x.shape[1] == CONFIG["input_dim"], f"Dim mismatch: {s0.x.shape[1]} != {CONFIG['input_dim']}"

# Save final dataset (versioned name + legacy alias)
for _ds_path in [OUTPUT_DIR / "pyg_dataset_v2.pt",
                 OUTPUT_DIR / "juliet_graphs_v2.pt"]:
    try:
        torch.save(pyg_dataset, str(_ds_path))
        print(f"  [CKPT] Saved {len(pyg_dataset)} graphs -> {_ds_path}")
    except Exception as _sv_err:
        print(f"  [CKPT] Save failed ({_ds_path}): {_sv_err}")
# Remove partial checkpoint now that the full dataset is safe
try:
    if _pyg_ckpt.exists():
        _pyg_ckpt.unlink()
        print(f"  [CKPT] Removed partial checkpoint")
except Exception:
    pass

Loading GraphCodeBERT...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  Loaded on cuda (pre-trained: Python, JS, Java, Go, PHP, Ruby)

Building PyG dataset (11339 samples, starting at 0)...
  Features: 768 + 6 = 774


Building graphs: 100%|██████████| 11339/11339 [46:08<00:00,  4.10it/s] 



  Built 11339 graphs, 0 failed
  Freed GraphCodeBERT from GPU
  Sample: 48 nodes, 95 edges, dim=774
  [CKPT] Saved 11339 graphs -> /kaggle/working/pyg_dataset_v2.pt
  [CKPT] Saved 11339 graphs -> /kaggle/working/juliet_graphs_v2.pt
  [CKPT] Removed partial checkpoint


## Cell 8: Dataset Split

In [8]:
# ============================================================================
# Cell 8: Stratified Dataset Split
# ============================================================================
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

strat_keys = [f"{d.y.item()}_{getattr(d, 'language', 'unk')}" for d in pyg_dataset]
indices = list(range(len(pyg_dataset)))
cal_test_ratio = CONFIG["cal_ratio"] + CONFIG["test_ratio"]

try:
    train_val_idx, cal_test_idx = train_test_split(
        indices, test_size=cal_test_ratio,
        stratify=[strat_keys[i] for i in indices], random_state=42)
except ValueError:
    print("  Stratification failed, using random split")
    train_val_idx, cal_test_idx = train_test_split(
        indices, test_size=cal_test_ratio, random_state=42)

val_frac = CONFIG["val_ratio"] / (CONFIG["train_ratio"] + CONFIG["val_ratio"])
try:
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_frac,
        stratify=[strat_keys[i] for i in train_val_idx], random_state=42)
except ValueError:
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_frac, random_state=42)

test_frac = CONFIG["test_ratio"] / (CONFIG["cal_ratio"] + CONFIG["test_ratio"])
try:
    cal_idx, test_idx = train_test_split(
        cal_test_idx, test_size=test_frac,
        stratify=[strat_keys[i] for i in cal_test_idx], random_state=42)
except ValueError:
    cal_idx, test_idx = train_test_split(
        cal_test_idx, test_size=test_frac, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data = [pyg_dataset[i] for i in val_idx]
cal_data = [pyg_dataset[i] for i in cal_idx]
test_data = [pyg_dataset[i] for i in test_idx]

print(f"Split (stratified by label + language):")
for name, split in [("Train", train_data), ("Val", val_data),
                     ("Cal", cal_data), ("Test", test_data)]:
    labels = [d.y.item() for d in split]
    nv = sum(labels)
    ns = len(labels) - nv
    print(f"  {name:5s}: {len(split):5d} (vuln={nv}, safe={ns}, ratio={nv/max(ns,1):.2f}:1)")

train_loader = DataLoader(train_data, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_data, batch_size=CONFIG["batch_size"], shuffle=False)
cal_loader = DataLoader(cal_data, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_data, batch_size=CONFIG["batch_size"], shuffle=False)
print(f"\nDataLoaders ready (batch_size={CONFIG['batch_size']})")

Split (stratified by label + language):
  Train:  6803 (vuln=1958, safe=4845, ratio=0.40:1)
  Val  :  1701 (vuln=490, safe=1211, ratio=0.40:1)
  Cal  :  1701 (vuln=491, safe=1210, ratio=0.41:1)
  Test :  1134 (vuln=326, safe=808, ratio=0.40:1)

DataLoaders ready (batch_size=32)


## Cell 9: MiniGAT V2 Model

In [9]:
# ============================================================================
# Cell 9: MiniGATv2 Model
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class MiniGATv2(nn.Module):
    """SEC-C Mini-GAT V2: 774-dim input (768 GCB + 6 structural incl language_id).

    Architecture: Linear(774->256) -> GAT(256->256,4h) -> GAT(256->128,4h) -> Pool -> Heads
    """
    def __init__(self, input_dim=774, hidden_dim=256, output_dim=128,
                 num_heads_l1=4, num_heads_l2=4, dropout=0.3, num_classes=2):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.dropout_rate = dropout
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // num_heads_l1,
                            heads=num_heads_l1, concat=True, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, output_dim // num_heads_l2,
                            heads=num_heads_l2, concat=True, dropout=dropout)
        self.dropout_layer = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_dim, num_classes)
        self.confidence_head = nn.Sequential(nn.Linear(output_dim, 1), nn.Sigmoid())
        self._attn_weights_l1 = None
        self._attn_weights_l2 = None

    def forward(self, x, edge_index, batch):
        h = F.relu(self.input_proj(x))
        h, a1 = self.gat1(h, edge_index, return_attention_weights=True)
        self._attn_weights_l1 = a1[1]
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout_rate, training=self.training)
        h, a2 = self.gat2(h, edge_index, return_attention_weights=True)
        self._attn_weights_l2 = a2[1]
        h = F.relu(h)
        graph_emb = global_mean_pool(h, batch)
        logits = self.classifier(graph_emb)
        confidence = self.confidence_head(graph_emb)
        return logits, confidence.squeeze(-1)

    def parameter_count(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

model = MiniGATv2(
    input_dim=CONFIG["input_dim"], hidden_dim=CONFIG["hidden_dim"],
    output_dim=CONFIG["output_dim"], num_heads_l1=CONFIG["num_heads_l1"],
    num_heads_l2=CONFIG["num_heads_l2"], dropout=CONFIG["dropout"],
    num_classes=CONFIG["num_classes"],
).to(device)

pc = model.parameter_count()
print(f"MiniGATv2 on {device}")
print(f"  Parameters: {pc['trainable']:,}")
print(f"  {CONFIG['input_dim']} -> {CONFIG['hidden_dim']} -> {CONFIG['output_dim']} -> {CONFIG['num_classes']}")

MiniGATv2 on cuda
  Parameters: 298,243
  774 -> 256 -> 128 -> 2


## Cell 10: Training with Focal Loss

In [10]:
# ============================================================================
# Cell 10: Training with Focal Loss + Cosine Annealing
# ============================================================================
import copy, time
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

class FocalLoss(nn.Module):
    """FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)"""
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

train_labels = [d.y.item() for d in train_data]
n_safe_t = train_labels.count(0)
n_vuln_t = train_labels.count(1)
w_safe = len(train_labels) / (2 * max(n_safe_t, 1))
w_vuln = len(train_labels) / (2 * max(n_vuln_t, 1))
class_weights = torch.tensor([w_safe, w_vuln], dtype=torch.float32).to(device)
print(f"Class weights: safe={w_safe:.3f}, vuln={w_vuln:.3f}")

optimizer = Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
criterion = FocalLoss(gamma=CONFIG["focal_gamma"], weight=class_weights)

def compute_metrics(preds, labels):
    n = len(preds)
    if n == 0:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}
    tp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    fn = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)
    correct = sum(1 for p, l in zip(preds, labels) if p == l)
    acc = correct / n
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

def train_one_epoch():
    model.train()
    total_loss, nb = 0.0, 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        logits, confidence = model(data.x, data.edge_index, data.batch)
        cls_loss = criterion(logits, data.y)
        with torch.no_grad():
            correct = (logits.argmax(dim=-1) == data.y).float()
        conf_loss = F.binary_cross_entropy(confidence, correct)
        loss = cls_loss + CONFIG["confidence_loss_weight"] * conf_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip"])
        optimizer.step()
        total_loss += loss.item()
        nb += 1
    return total_loss / max(nb, 1)

@torch.no_grad()
def validate(loader):
    model.eval()
    total_loss, nb = 0.0, 0
    all_preds, all_labels = [], []
    for data in loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        total_loss += criterion(logits, data.y).item()
        nb += 1
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(data.y.cpu().tolist())
    return total_loss / max(nb, 1), compute_metrics(all_preds, all_labels)

print(f"\n{'='*70}")
print(f"  Training MiniGATv2 ({CONFIG['epochs']} epochs, patience={CONFIG['patience']})")
print(f"  Focal Loss (gamma={CONFIG['focal_gamma']}), Cosine Annealing LR")
print(f"{'='*70}\n")

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_precision": [],
           "val_recall": [], "val_f1": []}
best_val_loss = float('inf')
best_val_f1 = 0.0
best_epoch = 0
patience_ctr = 0
best_state = None
t0 = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch()
    scheduler.step()
    val_loss, vm = validate(val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(vm["accuracy"])
    history["val_precision"].append(vm["precision"])
    history["val_recall"].append(vm["recall"])
    history["val_f1"].append(vm["f1"])
    improved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_f1 = vm["f1"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_ctr = 0
        improved = " *"
    else:
        patience_ctr += 1
    lr = optimizer.param_groups[0]['lr']
    # Periodic checkpoint every 10 epochs + on best
    if improved or epoch % 10 == 0:
        try:
            _ep_ckpt = OUTPUT_DIR / f"mini_gat_v2_epoch{epoch:03d}.pt"
            torch.save(model.state_dict(), str(_ep_ckpt))
            # Prune old periodic checkpoints (keep last 2)
            if epoch % 10 == 0 and epoch > 20 and not improved:
                _old_ep = OUTPUT_DIR / f"mini_gat_v2_epoch{epoch-20:03d}.pt"
                if _old_ep.exists():
                    _old_ep.unlink()
        except Exception:
            pass  # never let checkpoint failure abort training
    if epoch % 5 == 0 or epoch <= 3 or improved:
        print(f"  Epoch {epoch:3d}/{CONFIG['epochs']}  "
              f"tl={train_loss:.4f} vl={val_loss:.4f} "
              f"F1={vm['f1']:.4f} P={vm['precision']:.4f} R={vm['recall']:.4f} "
              f"lr={lr:.2e}{improved}")
    if patience_ctr >= CONFIG["patience"]:
        print(f"\n  Early stopping at epoch {epoch}")
        break

elapsed = time.time() - t0
print(f"\n  Done in {elapsed/60:.1f} min. Best epoch {best_epoch} (F1={best_val_f1:.4f})")
if best_state:
    model.load_state_dict(best_state)
    print("  Restored best weights")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history["train_loss"], label="Train"); axes[0].plot(history["val_loss"], label="Val")
axes[0].axvline(best_epoch-1, color='g', ls='--', alpha=.5, label=f'Best({best_epoch})')
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=.3)
axes[1].plot(history["val_f1"], label="F1", color='purple')
axes[1].plot(history["val_precision"], label="Prec", alpha=.7)
axes[1].plot(history["val_recall"], label="Rec", alpha=.7)
axes[1].set_title("Val Metrics"); axes[1].legend(); axes[1].grid(True, alpha=.3)
axes[2].plot(history["val_acc"], label="Acc", color='teal')
axes[2].set_title("Val Accuracy"); axes[2].legend(); axes[2].grid(True, alpha=.3)
plt.suptitle("MiniGATv2 Training Curves", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "training_curves.png"), dpi=150)
plt.show()

Class weights: safe=0.702, vuln=1.737

  Training MiniGATv2 (60 epochs, patience=15)
  Focal Loss (gamma=2.0), Cosine Annealing LR

  Epoch   1/60  tl=0.2965 vl=0.1643 F1=0.5333 P=0.3653 R=0.9878 lr=9.99e-04 *
  Epoch   2/60  tl=0.2760 vl=0.1618 F1=0.5291 P=0.3605 R=0.9939 lr=9.97e-04 *
  Epoch   3/60  tl=0.2683 vl=0.1556 F1=0.5695 P=0.4046 R=0.9612 lr=9.94e-04 *
  Epoch   5/60  tl=0.2579 vl=0.1611 F1=0.5669 P=0.4060 R=0.9388 lr=9.83e-04
  Epoch  10/60  tl=0.2438 vl=0.1719 F1=0.5705 P=0.4094 R=0.9408 lr=9.33e-04
  Epoch  15/60  tl=0.2365 vl=0.1918 F1=0.5873 P=0.4344 R=0.9061 lr=8.54e-04

  Early stopping at epoch 18

  Done in 1.1 min. Best epoch 3 (F1=0.5695)
  Restored best weights


## Cell 11: Evaluation

In [12]:
# ============================================================================
# Cell 11: Comprehensive Evaluation
# ============================================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

CLASS_LABELS = ["safe", "vulnerable"]
model.eval()
test_preds, test_labels, test_probs, test_langs, test_cwes = [], [], [], [], []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = logits.argmax(dim=-1).cpu().tolist()
        test_preds.extend(preds)
        test_labels.extend(data.y.cpu().tolist())
        test_probs.append(probs)
        # Extract language from feature
        batch_np = data.batch.cpu().numpy()
        for b in range(logits.shape[0]):
            mask = batch_np == b
            nidx = np.where(mask)[0]
            if len(nidx) > 0:
                lf = data.x[nidx[0], -1].item()
                ln = "unknown"
                for name, lid in CONFIG["language_ids"].items():
                    if abs(lf - lid) < 0.05:
                        ln = name
                        break
                test_langs.append(ln)
            else:
                test_langs.append("unknown")
        # PyG batches string attrs as lists (one entry per graph in batch)
        _batch_cwes = data.cwe if hasattr(data, 'cwe') else None
        if _batch_cwes is None:
            test_cwes.extend(['unknown'] * logits.shape[0])
        elif isinstance(_batch_cwes, list):
            test_cwes.extend([str(c) if c is not None else 'unknown' for c in _batch_cwes])
        else:
            # Unbatched single value — repeat for each graph
            test_cwes.extend([str(_batch_cwes)] * logits.shape[0])

test_probs_np = np.concatenate(test_probs, axis=0)
test_metrics = compute_metrics(test_preds, test_labels)

print("="*70)
print("  TEST SET EVALUATION")
print("="*70)
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1:        {test_metrics['f1']:.4f}")
try:
    auc_roc = roc_auc_score(test_labels, test_probs_np[:, 1])
    print(f"  AUC-ROC:   {auc_roc:.4f}")
except Exception:
    auc_roc = 0.0
    print("  AUC-ROC:   N/A")

print(f"\n{classification_report(test_labels, test_preds, target_names=CLASS_LABELS)}")

# Per-language
print(f"\n  Per-Language:")
print(f"  {'Lang':<15} {'N':>5} {'Acc':>7} {'P':>7} {'R':>7} {'F1':>7}")
print(f"  {'-'*50}")
lang_metrics = {}
for lang in CONFIG["language_ids"]:
    mi = [i for i, l in enumerate(test_langs) if l == lang]
    if not mi:
        continue
    lp = [test_preds[i] for i in mi]
    ll = [test_labels[i] for i in mi]
    m = compute_metrics(lp, ll)
    lang_metrics[lang] = m
    print(f"  {lang:<15} {len(mi):>5} {m['accuracy']:>7.4f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f}")

# Per-CWE (top 15)
print(f"\n  Per-CWE (top 15):")
cwe_ctr = Counter(str(c) for c in test_cwes)
print(f"  {'CWE':<20} {'N':>5} {'F1':>7}")
print(f"  {'-'*35}")
for cwe, _ in cwe_ctr.most_common(15):
    ci = [i for i, c in enumerate(test_cwes) if str(c) == cwe]
    if len(ci) < 2:
        continue
    m = compute_metrics([test_preds[i] for i in ci], [test_labels[i] for i in ci])
    print(f"  {cwe:<20} {len(ci):>5} {m['f1']:>7.4f}")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(test_labels, test_preds)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title("Confusion Matrix"); axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(CLASS_LABELS); axes[0].set_yticklabels(CLASS_LABELS)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
plt.colorbar(im, ax=axes[0])
try:
    fpr, tpr, _ = roc_curve(test_labels, test_probs_np[:,1])
    axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc_roc:.4f}')
    axes[1].plot([0,1],[0,1],'r--',alpha=.5); axes[1].set_title("ROC"); axes[1].legend()
except Exception:
    axes[1].text(.5,.5,"N/A",ha='center')
try:
    pr, rc, _ = precision_recall_curve(test_labels, test_probs_np[:,1])
    axes[2].plot(rc, pr, 'g-', lw=2); axes[2].set_title("Precision-Recall")
except Exception:
    axes[2].text(.5,.5,"N/A",ha='center')
plt.suptitle("MiniGATv2 Evaluation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "evaluation_plots.png"), dpi=150)
plt.show()

  TEST SET EVALUATION
  Accuracy:  0.5705
  Precision: 0.3969
  Recall:    0.9509
  F1:        0.5601
  AUC-ROC:   0.7433

              precision    recall  f1-score   support

        safe       0.95      0.42      0.58       808
  vulnerable       0.40      0.95      0.56       326

    accuracy                           0.57      1134
   macro avg       0.68      0.68      0.57      1134
weighted avg       0.79      0.57      0.57      1134


  Per-Language:
  Lang                N     Acc       P       R      F1
  --------------------------------------------------
  python             54  0.4630  0.4630  1.0000  0.6329
  javascript         54  0.4815  0.4815  1.0000  0.6500
  java              138  0.8043  0.4186  0.9000  0.5714
  c_cpp             872  0.5482  0.3811  0.9435  0.5429
  go                 16  0.4375  0.4375  1.0000  0.6087

  Per-CWE (top 15):
  CWE                      N      F1
  -----------------------------------
  CWE-unknown            694  0.6223
  CWE-190  

## Cell 12: Conformal Prediction (APS, alpha=0.3)

In [13]:
# ============================================================================
# Cell 12: Conformal Prediction Calibration (APS)
# ============================================================================
import math

def compute_aps_scores(softmax_probs, true_labels):
    n = len(true_labels)
    scores = np.zeros(n, dtype=np.float64)
    for i in range(n):
        probs = softmax_probs[i]
        si = np.argsort(-probs)
        cs = np.cumsum(probs[si])
        rank = int(np.where(si == true_labels[i])[0][0])
        scores[i] = cs[rank]
    return scores

def build_pred_set(probs, threshold):
    si = np.argsort(-probs)
    cs = np.cumsum(probs[si])
    ps = []
    for j, idx in enumerate(si):
        ps.append(CLASS_LABELS[int(idx)])
        if cs[j] >= threshold:
            break
    return ps if ps else [CLASS_LABELS[int(si[0])]]

# Collect calibration probabilities
print(f"{'='*60}")
print(f"  Conformal Prediction (APS, alpha={CONFIG['alpha']})")
print(f"{'='*60}")

model.eval()
cal_sm, cal_lb = [], []
with torch.no_grad():
    for data in cal_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        cal_sm.append(F.softmax(logits, dim=-1).cpu().numpy())
        cal_lb.extend(data.y.cpu().tolist())

cal_sm = np.concatenate(cal_sm, axis=0)
cal_lb_np = np.array(cal_lb, dtype=np.int64)
n_cal = len(cal_lb_np)

alpha = CONFIG["alpha"]
scores = compute_aps_scores(cal_sm, cal_lb_np)
ql = min(math.ceil((n_cal + 1) * (1.0 - alpha)) / n_cal, 1.0)
try:
    threshold = float(np.quantile(scores, ql, method="higher"))
except TypeError:
    threshold = float(np.quantile(scores, ql, interpolation="higher"))

print(f"\n  Cal samples:    {n_cal}")
print(f"  Alpha:          {alpha}")
print(f"  Quantile:       {ql:.4f}")
print(f"  Threshold:      {threshold:.4f}")
print(f"  Score mean:     {scores.mean():.4f}")

# Evaluate on cal set
covered, single, ambig, sizes = 0, 0, 0, []
for i in range(n_cal):
    ps = build_pred_set(cal_sm[i], threshold)
    sizes.append(len(ps))
    if CLASS_LABELS[cal_lb_np[i]] in ps:
        covered += 1
    if len(ps) == 1:
        single += 1
    else:
        ambig += 1

empirical_coverage = covered / n_cal
singleton_rate = single / n_cal
ambiguous_rate = ambig / n_cal

print(f"\n  [Calibration]")
print(f"  Coverage:       {empirical_coverage:.4f} (target >= {1-alpha:.2f})")
print(f"  Singleton:      {single}/{n_cal} ({singleton_rate:.1%})")
print(f"  Ambiguous:      {ambig}/{n_cal} ({ambiguous_rate:.1%})")

# Verify on test
tc, ts, ta, tsz = 0, 0, 0, []
for i in range(len(test_labels)):
    ps = build_pred_set(test_probs_np[i], threshold)
    tsz.append(len(ps))
    if CLASS_LABELS[test_labels[i]] in ps:
        tc += 1
    if len(ps) == 1:
        ts += 1
    else:
        ta += 1
n_test = len(test_labels)
test_coverage = tc / n_test
test_singleton_rate = ts / n_test
test_ambiguous_rate = ta / n_test

print(f"\n  [Test Verification]")
print(f"  Coverage:       {test_coverage:.4f} (target >= {1-alpha:.2f})")
print(f"  Singleton:      {ts}/{n_test} ({test_singleton_rate:.1%})")
print(f"  Ambiguous:      {ta}/{n_test} ({test_ambiguous_rate:.1%})")
coverage_ok = test_coverage >= (1 - alpha)
print(f"  Guarantee:      {'MET' if coverage_ok else 'NOT MET'}")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(scores, bins=30, color="#4dabf7", alpha=.7, edgecolor="black")
axes[0].axvline(threshold, color="red", ls="--", lw=2, label=f"Thr={threshold:.3f}")
axes[0].set_title("APS Scores (Cal)"); axes[0].legend()

sc = Counter(sizes)
su = sorted(sc.keys())
cv = [sc[s] for s in su]
cl = ["#51cf66", "#ffa94d", "#ff6b6b"]
bars = axes[1].bar(su, cv, color=cl[:len(su)])
axes[1].set_title("Prediction Set Sizes")
for b, c in zip(bars, cv):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1, str(c), ha="center", fontweight="bold")

alphas_t = np.linspace(0.01, 0.5, 50)
covs = []
for a in alphas_t:
    q = min(math.ceil((n_cal+1)*(1.0-a))/n_cal, 1.0)
    try:
        t = float(np.quantile(scores, q, method="higher"))
    except TypeError:
        t = float(np.quantile(scores, q, interpolation="higher"))
    c = sum(1 for i in range(n_cal) if CLASS_LABELS[cal_lb_np[i]] in build_pred_set(cal_sm[i], t))
    covs.append(c / n_cal)
axes[2].plot(alphas_t, covs, "b-", lw=2, label="Empirical")
axes[2].plot(alphas_t, 1-alphas_t, "r--", alpha=.7, label="1-alpha")
axes[2].axvline(alpha, color="green", ls=":", label=f"alpha={alpha}")
axes[2].set_title("Calibration Curve"); axes[2].legend(); axes[2].grid(True, alpha=.3)

plt.suptitle("Conformal Prediction Diagnostics (V2)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "conformal_diagnostics.png"), dpi=150)
plt.show()

  Conformal Prediction (APS, alpha=0.3)

  Cal samples:    1701
  Alpha:          0.3
  Quantile:       0.7008
  Threshold:      1.0000
  Score mean:     0.8244

  [Calibration]
  Coverage:       1.0000 (target >= 0.70)
  Singleton:      0/1701 (0.0%)
  Ambiguous:      1701/1701 (100.0%)

  [Test Verification]
  Coverage:       1.0000 (target >= 0.70)
  Singleton:      0/1134 (0.0%)
  Ambiguous:      1134/1134 (100.0%)
  Guarantee:      MET


## Cell 13: Export Artifacts

In [14]:
# ============================================================================
# Cell 13: Export Artifacts
# ============================================================================
import json
import torch_geometric

# 1. Model
model_path = OUTPUT_DIR / "mini_gat_v2.pt"
torch.save(model.cpu().state_dict(), str(model_path))
model_size = model_path.stat().st_size
print(f"1. Model:       {model_path}  ({model_size/1024:.0f} KB)")

# 2. Conformal calibration
cal_export = {
    "alpha": CONFIG["alpha"],
    "threshold": float(threshold),
    "n_calibration": int(n_cal),
    "empirical_coverage": float(empirical_coverage),
    "test_coverage": float(test_coverage),
    "class_names": CLASS_LABELS,
    "singleton_rate": float(singleton_rate),
    "ambiguous_rate": float(ambiguous_rate),
    "test_singleton_rate": float(test_singleton_rate),
    "test_ambiguous_rate": float(test_ambiguous_rate),
    "mean_set_size": float(np.mean(sizes)),
    "score_stats": {
        "mean": float(scores.mean()), "std": float(scores.std()),
        "median": float(np.median(scores)),
        "min": float(scores.min()), "max": float(scores.max()),
    },
    "test_metrics": {
        "accuracy": float(test_metrics["accuracy"]),
        "precision": float(test_metrics["precision"]),
        "recall": float(test_metrics["recall"]),
        "f1": float(test_metrics["f1"]),
        "auc_roc": float(auc_roc),
    },
    "per_language_metrics": {
        l: {k: float(v) for k, v in m.items()} for l, m in lang_metrics.items()
    },
    "training_history": {
        "total_epochs": len(history["train_loss"]),
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "best_val_f1": float(best_val_f1),
    },
    "language_map": CONFIG["language_ids"],
    "version": "v2",
}
cal_path = OUTPUT_DIR / "conformal_calibration_v2.json"
with open(str(cal_path), "w") as f:
    json.dump(cal_export, f, indent=2)
print(f"2. Calibration: {cal_path}")

# 3. Graph config
graph_config = {
    "model_config": {
        "input_dim": CONFIG["input_dim"], "hidden_dim": CONFIG["hidden_dim"],
        "output_dim": CONFIG["output_dim"],
        "num_heads_l1": CONFIG["num_heads_l1"], "num_heads_l2": CONFIG["num_heads_l2"],
        "dropout": CONFIG["dropout"], "num_classes": CONFIG["num_classes"],
        "embedding_model": CONFIG["embedding_model"],
        "embedding_dim": CONFIG["embedding_dim"],
        "max_nodes": CONFIG["max_nodes"],
        "focal_gamma": CONFIG["focal_gamma"],
    },
    "node_features": [
        "in_degree_norm", "out_degree_norm", "is_sink",
        "is_source", "depth_norm", "language_id",
    ],
    "language_ids": CONFIG["language_ids"],
    "sink_patterns": SINK_PATTERNS,
    "source_patterns": SOURCE_PATTERNS,
    "max_nodes": CONFIG["max_nodes"],
    "embedding_model": CONFIG["embedding_model"],
    "torch_version": torch.__version__,
    "torch_geometric_version": torch_geometric.__version__,
    "dataset_info": {
        "total_samples": len(pyg_dataset),
        "train_samples": len(train_data), "val_samples": len(val_data),
        "cal_samples": len(cal_data), "test_samples": len(test_data),
        "languages": list(CONFIG["language_ids"].keys()),
        "sources": list(set(s.get("source", "unknown") for s in all_samples)),
    },
    "version": "v2",
}
config_path = OUTPUT_DIR / "graph_config_v2.json"
with open(str(config_path), "w") as f:
    json.dump(graph_config, f, indent=2)
print(f"3. Config:      {config_path}")
# Legacy aliases for backward compatibility
import shutil as _shutil
for _alias in ["conformal_calibration.json", "graph_config.json"]:
    _alias_src = OUTPUT_DIR / _alias.replace(".json", "_v2.json")
    _alias_dst = OUTPUT_DIR / _alias
    if _alias_src.exists() and not _alias_dst.exists():
        try: _shutil.copy(str(_alias_src), str(_alias_dst))
        except Exception: pass

# Backward compat copy
import shutil
compat = OUTPUT_DIR / "mini_gat.pt"
if not compat.exists():
    shutil.copy(str(model_path), str(compat))
    print(f"   (Copied as {compat})")

print(f"\nAll artifacts in {OUTPUT_DIR}")

1. Model:       /kaggle/working/mini_gat_v2.pt  (1170 KB)
2. Calibration: /kaggle/working/conformal_calibration_v2.json
3. Config:      /kaggle/working/graph_config_v2.json
   (Copied as /kaggle/working/mini_gat.pt)

All artifacts in /kaggle/working


## Cell 14: Summary Report

In [15]:
# ============================================================================
# Cell 14: Final Summary Report
# ============================================================================
print("="*70)
print("  SEC-C MiniGATv2 -- Training Report")
print("="*70)
print(f"\n  MODEL:  MiniGATv2, {pc['trainable']:,} params")
print(f"  INPUT:  {CONFIG['input_dim']} (768 GCB + 6 structural)")
print(f"  ARCH:   {CONFIG['input_dim']} -> {CONFIG['hidden_dim']} -> {CONFIG['output_dim']} -> 2")
print(f"\n  DATA:   {len(pyg_dataset)} graphs, {', '.join(CONFIG['language_ids'].keys())}")
print(f"  SPLIT:  {len(train_data)}/{len(val_data)}/{len(cal_data)}/{len(test_data)}")
sources_used = sorted(set(s.get("source","?") for s in all_samples))
print(f"  SRCS:   {', '.join(sources_used)}")
print(f"\n  TRAIN:  Focal(gamma={CONFIG['focal_gamma']}), CosineAnnealing, patience={CONFIG['patience']}")
print(f"  BEST:   epoch {best_epoch}, val_loss={best_val_loss:.4f}, val_F1={best_val_f1:.4f}")
print(f"\n  TEST METRICS:")
print(f"    Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"    Precision: {test_metrics['precision']:.4f}")
print(f"    Recall:    {test_metrics['recall']:.4f}")
print(f"    F1:        {test_metrics['f1']:.4f}")
print(f"    AUC-ROC:   {auc_roc:.4f}")
print(f"\n  PER-LANGUAGE F1:")
for lang, m in lang_metrics.items():
    print(f"    {lang:<15} {m['f1']:.4f}")
print(f"\n  CONFORMAL (alpha={CONFIG['alpha']}):")
print(f"    Threshold:    {threshold:.4f}")
print(f"    Test coverage: {test_coverage:.4f} (>= {1-alpha:.2f}: {'YES' if coverage_ok else 'NO'})")
print(f"    Singleton:    {test_singleton_rate:.1%}")
print(f"    Ambiguous:    {test_ambiguous_rate:.1%}")
print(f"\n  V1 -> V2:")
print(f"    Languages:  2 -> 5")
print(f"    Datasets:   Juliet -> CVEfixes+DiverseVul+Devign+Juliet+synthetic")
print(f"    Balance:    13:1 -> ~{total_vuln/max(total_safe,1):.1f}:1")
print(f"    Edges:      sequential -> AST+CFG+DDG")
print(f"    Features:   773 -> 774 (+language_id)")
print(f"    Loss:       WeightedCE -> Focal(2.0)")
print(f"    Alpha:      0.1 -> 0.3")
print(f"\n  OUTPUT FILES:")
print(f"    mini_gat_v2.pt, conformal_calibration.json, graph_config.json")
print("="*70)

  SEC-C MiniGATv2 -- Training Report

  MODEL:  MiniGATv2, 298,243 params
  INPUT:  774 (768 GCB + 6 structural)
  ARCH:   774 -> 256 -> 128 -> 2

  DATA:   11339 graphs, python, javascript, java, c_cpp, go
  SPLIT:  6803/1701/1701/1134
  SRCS:   CVEfixes, Devign, DiverseVul, Juliet-C, Juliet-Java

  TRAIN:  Focal(gamma=2.0), CosineAnnealing, patience=15
  BEST:   epoch 3, val_loss=0.1556, val_F1=0.5695

  TEST METRICS:
    Accuracy:  0.5705
    Precision: 0.3969
    Recall:    0.9509
    F1:        0.5601
    AUC-ROC:   0.7433

  PER-LANGUAGE F1:
    python          0.6329
    javascript      0.6500
    java            0.5714
    c_cpp           0.5429
    go              0.6087

  CONFORMAL (alpha=0.3):
    Threshold:    1.0000
    Test coverage: 1.0000 (>= 0.70: YES)
    Singleton:    0.0%
    Ambiguous:    100.0%

  V1 -> V2:
    Languages:  2 -> 5
    Datasets:   Juliet -> CVEfixes+DiverseVul+Devign+Juliet+synthetic
    Balance:    13:1 -> ~0.4:1
    Edges:      sequential -> AST+